# Using the DynamicDecorator Module in baseobjects

## Introduction

DynamicDecorator combines the decorator-centric features of BaseDecorator with the multiplexed binding and callback capabilities of DynamicFunction (via MethodMultiplexer). In short, it is a decorator class whose descriptor binding (__get__) and call behavior (__call__) can be switched at runtime.

This tutorial highlights:
- How DynamicDecorator differs from BaseDecorator
- How binding and callback are delegated to MethodMultiplexer
- How to switch bind_method and call_method dynamically
- How to use DynamicDecorator for both simple and parameterized decorators

**Prerequisites:**
- Familiarity with BaseDecorator and DynamicCallable
- Understanding of the descriptor protocol and Python decorators

### Table of Contents
- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting--FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)


## Importing the Module

In [11]:
from baseobjects.functions import DynamicDecorator, MethodMultiplexer


## Core Functionality

DynamicDecorator is an abstract decorator class that:
- Inherits decorator behavior from BaseDecorator (can be used as @Decorator or @Decorator(...)).
- Inherits multiplexed binding and calling from DynamicFunction.
- Exposes two MethodMultiplexer instances:
  - bind_multiplexer: controls descriptor binding (__get__).
  - call_multiplexer: controls call behavior (__call__).
- Lets you select strategies via bind_method and call_method properties, or during construction.

Below we implement a concrete subclass to demonstrate switching strategies for both binding and calling.


In [12]:
class DemoDynamicDecorator(DynamicDecorator):
    """Demo subclass showcasing multiplexed bind and call for decorators."""

    # Example call strategies selectable by call_multiplexer
    def call_wrapped(self, *args, **kwargs):
        # Neutral call path: just call the wrapped function
        return self.__wrapped__(*args, **kwargs)

    def call_with_logging(self, *args, **kwargs):
        print(f"[log] calling {self.__wrapped__.__name__} with args={args}, kwargs={kwargs}")
        result = self.__wrapped__(*args, **kwargs)
        print(f"[log] result: {result!r}")
        return result

    def call_upper_string(self, *args, **kwargs):
        # If result is a string, uppercase it
        result = self.__wrapped__(*args, **kwargs)
        if isinstance(result, str):
            return result.upper()
        return result

    # Example binding strategies selectable by bind_multiplexer
    def bind_builtin(self, instance=None, owner=None):
        # Default binding behavior (delegates to the underlying base implementation)
        return super().bind_builtin(instance=instance, owner=owner)

    def bind_self(self, instance=None, owner=None):
        # Alternate binding that simply returns this decorator instance
        # (for demonstration; in real scenarios, you'd return an appropriate bound object)
        return self

# Create a parameterless decorator instance
LogOrUpper = DemoDynamicDecorator

# Use default call strategy (often mapped to call_wrapped)
@LogOrUpper
def greet(name):
    return f"Hello, {name}!"

print("Default strategy:", greet("World"))

# Switch strategy at runtime to logging
greet.call_method = "call_with_logging"
print("Logging strategy:", greet("Alice"))

Default strategy: Hello, World!
[log] calling greet with args=('Alice',), kwargs={}
[log] result: 'Hello, Alice!'
Logging strategy: Hello, Alice!


### Method Multiplexer Highlight: Binding vs Callback

- bind_multiplexer governs descriptor binding (__get__), which affects how the decorator instance behaves when used as a class attribute.
- call_multiplexer governs callback (__call__), the path taken when the decorated function is invoked.

You can select methods by name:
- Using properties: obj.bind_method = "..." and obj.call_method = "..."
- During construction: DemoDynamicDecorator(None, bind_method="...", call_method="...")

The following demonstrates descriptor binding differences when the decorator is stored as a class attribute.


In [13]:
class Container:
    # Store an instance as a class attribute to showcase binding
    @DemoDynamicDecorator
    def do(self, x):
        return f"do({x}) by {self.__class__.__name__}"

c = Container()

# Default binding
bound1 = c.do
print("Default binding type:", type(bound1))

# Switch binding strategy and rebind
Container.do.bind_method = "bind"  # Note: Calling "do" from Class guarantees the correct that "do" is not a bound method
bound2 = c.do
print("With bind type:", type(bound2))

# Call the method to ensure decoration still works
print(c.do(5))


Default binding type: <class 'method'>
With bind type: <class 'baseobjects.functions.dynamiccallable.DynamicMethod'>
do(5) by Container


## Module Interaction

DynamicDecorator uses MethodMultiplexer under the hood to manage strategy selection for binding and calling. It also inherits the decorator semantics from BaseDecorator, and the multiplexed mechanics from DynamicFunction. The multiplexer maintains:
- A registry of available strategies (method names available on the instance)
- The currently selected strategy key for both binding and calling


## Advanced Features

- Preselect strategies during construction using construct arguments.
- Use parameterized decorators (decorators with arguments) and still benefit from multiplexing.


In [14]:
# Preselect strategies at creation time
@DemoDynamicDecorator(None, bind_method="bind_builtin", call_method="call_upper_string")
def shout(msg):
    return msg

print(shout("Dynamic at construct time"))

# Parameterized decorator using subclass init
class TaggingDecorator(DemoDynamicDecorator):
    def __init__(self, func=None, prefix="", suffix=""):
        super().__init__(func)
        self.prefix = prefix
        self.suffix = suffix

    def call_wrapped(self, *args, **kwargs):
        result = self.__wrapped__(*args, **kwargs)
        if isinstance(result, str):
            return f"{self.prefix}{result}{self.suffix}"
        return result

# Use parameterized mode
@TaggingDecorator(prefix="[START] ", suffix=" [END]")
def process(text):
    return text.upper()

print(process("hello"))

# Switch call strategy at runtime to logging
process.call_method = "call_with_logging"
print(process("world"))


DYNAMIC AT CONSTRUCT TIME
[START] HELLO [END]
[log] calling process with args=('world',), kwargs={}
[log] result: 'WORLD'
WORLD


## Examples

- Toggle call strategies at runtime for instrumentation (logging, metrics, feature flags).
- Choose different binding behavior for special descriptor use-cases (e.g., return a stable instance vs. a Python bound method).


In [15]:
# Example: switch strategies around a computational function
class MetricDecorator(DemoDynamicDecorator):
    def call_with_metrics(self, *args, **kwargs):
        import time
        t0 = time.time()
        result = self.__wrapped__(*args, **kwargs)
        dt = time.time() - t0
        print(f"[metrics] {self.__wrapped__.__name__} took {dt:.4f}s")
        return result

@MetricDecorator
def fib(n):
    return 1 if n <= 2 else fib(n-1) + fib(n-2)

# default path
print(fib(10))

# switch to metrics
fib.call_method = "call_with_metrics"
print(fib(10))


55
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0005s
[metrics] fib took 0.0000s
[metrics] fib took 0.0005s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0005s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000s
[metrics] fib took 0.0000

## API Highlights

- Properties
  - bind_method: currently selected binding strategy name
  - call_method: currently selected call strategy name
- Attributes
  - bind_multiplexer: MethodMultiplexer controlling __get__
  - call_multiplexer: MethodMultiplexer controlling __call__
- Constructor / construct
  - construct(func=None, *, bind_method=None, call_method=None, **kwargs)
- Behavior
  - __get__ delegates to bind_multiplexer
  - __call__ delegates to call_multiplexer

Refer to the full API documentation for details and additional strategies available in your version.


## Troubleshooting / FAQs

### Q: My custom strategy name isn't being used.
A: Ensure the method exists on the decorator instance and that you set the exact string on bind_method or call_method. Also confirm you are mutating the same decorator instance that wraps the function.

### Q: Pickling loses my selected strategy.
A: DynamicDecorator persists its multiplexer state via its serialization hooks. If you observe issues, verify your environment's versions and that you're pickling the decorator instance, not the decorated function.

### Q: How do I make a parameterized decorator using DynamicDecorator?
A: Provide an __init__(self, func=None, ...) that stores parameters, then implement call_wrapped (or another call_* strategy) to use those parameters. DynamicDecorator will manage dual-mode behavior via BaseDecorator.


## Conclusion and Next Steps

You learned how DynamicDecorator uses MethodMultiplexer to control both binding (__get__) and callback (__call__) behavior while retaining full decorator capabilities. This enables runtime strategy switching for instrumentation, formatting, or alternate binding semantics.

Next steps:
- Explore DynamicFunction and DynamicMethod for specialized semantics built on the same multiplexing foundation.
- Review MethodMultiplexer tutorial for deeper control of strategy registration and selection.
- Create your own strategy methods (bind_* and call_*) to tailor behavior to your use-cases.
